# Section 9 — Classification

**Input files (from preprocessing):**
- `X_train_preprocessed.csv` — training features
- `y_train.csv` — training labels
- `X_test_preprocessed.csv` — test features

**Steps:**
1. Load data
2. Check class distribution
3. Train / Validation split
4. Models — Logistic Regression, Decision Tree, Random Forest, Naive Bayes, SVM
5. 10-Fold Stratified Cross-Validation
6. Confusion Matrices + ROC Curves
7. Hyperparameter Tuning (GridSearchCV)
8. Imbalanced data handling — SMOTE, ADASYN, RandomUnderSampler
9. Cost-Sensitive Learning — class_weight + threshold
10. Final evaluation + predictions on test set

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer

from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.ensemble import RandomForestClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression

from sklearn.model_selection import (
    train_test_split, cross_val_score,
    GridSearchCV, StratifiedKFold,
)
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    classification_report, confusion_matrix, ConfusionMatrixDisplay,
    roc_curve, roc_auc_score, precision_recall_curve, average_precision_score,
)

try:
    from imblearn.over_sampling import SMOTE, ADASYN
    from imblearn.under_sampling import RandomUnderSampler
    IMBLEARN_AVAILABLE = True
    print('imbalanced-learn loaded.')
except ImportError:
    IMBLEARN_AVAILABLE = False
    print('imbalanced-learn not installed.')
    print('To install:  pip install imbalanced-learn')

import warnings
warnings.filterwarnings('ignore')

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

%matplotlib inline
plt.rcParams['figure.dpi'] = 100
try:
    sns.set_theme(style='whitegrid', palette='muted')
except AttributeError:
    sns.set(style='whitegrid', palette='muted')

## Step 9.1 — Load Preprocessed Data

Load the CSV files produced at the end of the preprocessing stage.

In [ ]:
X_train_full = pd.read_csv('PreProcessedData/X_train_preprocessed.csv')
y_train_full = pd.read_csv('PreProcessedData/y_train.csv').squeeze()
X_test       = pd.read_csv('PreProcessedData/X_test_preprocessed.csv')

print(f'X_train_full : {X_train_full.shape}')
print(f'y_train_full : {y_train_full.shape}')
print(f'X_test       : {X_test.shape}')
print(f'\nFeatures: {X_train_full.columns.tolist()}')

In [ ]:
# Categorical features encoded as integers during preprocessing.
# These are identifiers, not continuous values — models that assume
# continuous inputs (LR, NB, SVM) receive OHE + scaling via a Pipeline.
# Tree-based models (DT, RF) use the raw representation.
CAT_COLS = ['month', 'browser', 'region', 'traffic_type', 'is_weekend', 'visitor_type']
NUM_COLS = [c for c in X_train_full.columns if c not in CAT_COLS]

print(f'Categorical columns ({len(CAT_COLS)}): {CAT_COLS}')
print(f'Numeric columns    ({len(NUM_COLS)}): {NUM_COLS}')


def make_preprocessor():
    """ColumnTransformer: OHE for encoded categoricals + StandardScaler for numerics.
    Applied inside individual model pipelines — preprocessing CSVs are not touched."""
    try:
        ohe = OneHotEncoder(handle_unknown='ignore', sparse_output=False)
    except TypeError:                      # sklearn < 1.2 uses sparse=
        ohe = OneHotEncoder(handle_unknown='ignore', sparse=False)
    return ColumnTransformer(
        transformers=[
            ('cat', ohe,              CAT_COLS),
            ('num', StandardScaler(), NUM_COLS),
        ]
    )

## Step 9.2 — Class Distribution

Before building models, we examine the distribution of the target variable `high_intent`.

If there is class imbalance, Accuracy alone is not a sufficient metric — we use F1, Recall, and ROC-AUC.

In [ ]:
print('Class distribution:')
print(y_train_full.value_counts())
print(f'\nPositive class rate : {y_train_full.mean():.1%}')
print(f'Ratio (0:1)         : {(y_train_full==0).sum()} : {(y_train_full==1).sum()}')

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

counts = y_train_full.value_counts().sort_index()
sns.countplot(x=y_train_full, ax=axes[0], palette=['#4C72B0', '#DD8452'])
axes[0].set_title('Class Count', fontsize=12)
axes[0].set_xlabel('high_intent')
axes[0].set_ylabel('Count')
axes[0].set_xticks([0, 1])
axes[0].set_xticklabels(['Low Intent (0)', 'High Intent (1)'])
for bar in axes[0].patches:
    axes[0].text(bar.get_x() + bar.get_width()/2,
                 bar.get_height() + 20,
                 str(int(bar.get_height())), ha='center', fontsize=11)

axes[1].pie(counts, labels=['Low Intent (0)', 'High Intent (1)'],
            autopct='%1.1f%%', colors=['#4C72B0', '#DD8452'], startangle=90)
axes[1].set_title('Class Proportion', fontsize=12)

plt.suptitle('Target Variable: high_intent', fontsize=14)
plt.tight_layout()
plt.show()

minority_rate = y_train_full.mean()
if minority_rate < 0.4:
    print(f'\nThe dataset is imbalanced ({minority_rate:.1%} positive class).')
    print('Primary metrics: F1, Recall, ROC-AUC.')

## Step 9.3 — Train / Validation Split

The training set is split into 80% train and 20% validation.

`stratify=y` preserves the same class distribution in both sets — especially important when the dataset is imbalanced.

In [ ]:
X_train, X_val, y_train, y_val = train_test_split(
    X_train_full, y_train_full,
    test_size=0.2,
    stratify=y_train_full,
    random_state=RANDOM_SEED,
)

print(f'Train set : {X_train.shape}  |  Positive rate: {y_train.mean():.1%}')
print(f'Val set   : {X_val.shape}    |  Positive rate: {y_val.mean():.1%}')
print(f'Test set  : {X_test.shape}')
print('\nstratify=True preserved the class ratio in both splits.')

## Step 9.4 — Candidate Models

Five classifiers are defined:

| Model | Type |
|---|---|
| Logistic Regression | Linear probabilistic model |
| Decision Tree | Decision tree — splits by Gini / Information Gain |
| Random Forest | Ensemble of trees — Majority Vote |
| Naive Bayes | Based on Bayes' theorem with independence assumption |
| SVM | Maximises margin between classes, supports non-linear kernels |

In [ ]:
# Logistic Regression, Naive Bayes, and SVM are wrapped in a Pipeline that
# applies OHE + StandardScaler to the encoded categorical columns and scales
# numeric columns.  This is done here — not in preprocessing — because the
# preprocessing CSVs are frozen.
#
# Decision Tree and Random Forest receive the raw encoded representation,
# which is appropriate for tree-based splitters (they treat each value as
# a separate category implicitly through threshold splits).

models = {
    'Logistic Regression': Pipeline([
        ('pre', make_preprocessor()),
        ('clf', LogisticRegression(max_iter=1000, random_state=RANDOM_SEED)),
    ]),
    'Decision Tree': DecisionTreeClassifier(random_state=RANDOM_SEED),
    'Random Forest': RandomForestClassifier(n_estimators=100,
                                             random_state=RANDOM_SEED, n_jobs=-1),
    'Naive Bayes': Pipeline([
        ('pre', make_preprocessor()),
        ('clf', GaussianNB()),
    ]),
    'SVM': Pipeline([
        ('pre', make_preprocessor()),
        ('clf', SVC(probability=True, kernel='rbf', random_state=RANDOM_SEED)),
    ]),
}

print('Candidate models defined:')
for name, model in models.items():
    tag = ' (Pipeline: OHE + StandardScaler)' if isinstance(model, Pipeline) else ''
    print(f'  - {name}{tag}')
print('\nNote: SVM with RBF kernel may take several minutes on large datasets.')

## Step 9.5 — 10-Fold Stratified Cross-Validation

10-fold CV provides a more stable estimate than a single train/validation split.

Primary metrics: F1 and ROC-AUC (not Accuracy alone).

In [ ]:
CV_10 = StratifiedKFold(n_splits=10, shuffle=True, random_state=RANDOM_SEED)
cv_results = {}

print('10-Fold Stratified Cross-Validation\n')
for name, model in models.items():
    acc = cross_val_score(model, X_train_full, y_train_full,
                          cv=CV_10, scoring='accuracy', n_jobs=-1)
    f1  = cross_val_score(model, X_train_full, y_train_full,
                          cv=CV_10, scoring='f1', n_jobs=-1)
    auc = cross_val_score(model, X_train_full, y_train_full,
                          cv=CV_10, scoring='roc_auc', n_jobs=-1)
    cv_results[name] = {
        'Accuracy (mean)': round(acc.mean(), 4),
        'Accuracy (std)' : round(acc.std(),  4),
        'F1 (mean)'      : round(f1.mean(),  4),
        'ROC-AUC (mean)' : round(auc.mean(), 4),
    }
    print(f'  {name:<22}  Acc={acc.mean():.3f}±{acc.std():.3f}  '
          f'F1={f1.mean():.3f}  AUC={auc.mean():.3f}')

cv_df = pd.DataFrame(cv_results).T
print()
display(cv_df)

## Step 9.6 — Validation Set Evaluation

Each model is trained on the train set and evaluated on the validation set. Metrics: Accuracy, Precision, Recall, F1, ROC-AUC.

In [ ]:
eval_results = {}

for name, model in models.items():
    model.fit(X_train, y_train)
    y_pred  = model.predict(X_val)
    y_proba = model.predict_proba(X_val)[:, 1] if hasattr(model, 'predict_proba') else None

    eval_results[name] = {
        'Accuracy' : accuracy_score(y_val, y_pred),
        'Precision': precision_score(y_val, y_pred, zero_division=0),
        'Recall'   : recall_score(y_val, y_pred, zero_division=0),
        'F1'       : f1_score(y_val, y_pred, zero_division=0),
        'ROC-AUC'  : roc_auc_score(y_val, y_proba) if y_proba is not None else float('nan'),
    }

eval_df = pd.DataFrame(eval_results).T.round(4)
print('=== Validation Set Results — All Models ===')
display(eval_df)

In [ ]:
n = len(models)
fig, axes = plt.subplots(1, n, figsize=(5 * n, 4))

for ax, (name, model) in zip(axes, models.items()):
    y_pred = model.predict(X_val)
    cm = confusion_matrix(y_val, y_pred)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
                xticklabels=['Pred 0', 'Pred 1'],
                yticklabels=['True 0', 'True 1'])
    ax.set_title(name, fontsize=10)

plt.suptitle('Confusion Matrices — All Models', fontsize=13)
plt.tight_layout()
plt.show()

print('Confusion Matrix Terms:')
print('  TP (True Positive)  — predicted High Intent, actually High Intent')
print('  TN (True Negative)  — predicted Low Intent, actually Low Intent')
print('  FP (False Positive) — predicted High Intent, actually Low Intent  (Type I error)')
print('  FN (False Negative) — predicted Low Intent, actually High Intent  (Type II error)')

## Step 9.7 — ROC Curves

ROC curves for all models are plotted on the same axes. An AUC close to 1 indicates a good model.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 6))

for name, model in models.items():
    if hasattr(model, 'predict_proba'):
        y_proba = model.predict_proba(X_val)[:, 1]
        fpr, tpr, _ = roc_curve(y_val, y_proba)
        auc_val = roc_auc_score(y_val, y_proba)
        ax.plot(fpr, tpr, lw=2, label=f'{name}  (AUC = {auc_val:.3f})')

ax.plot([0, 1], [0, 1], 'k--', lw=1, label='Random classifier')
ax.set_xlabel('False Positive Rate', fontsize=11)
ax.set_ylabel('True Positive Rate', fontsize=11)
ax.set_title('ROC Curves — All Models', fontsize=13)
ax.legend(loc='lower right', fontsize=9)
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for ax, metric in zip(axes, ['Accuracy', 'F1', 'ROC-AUC']):
    vals = eval_df[metric].sort_values(ascending=False)
    bar_colors = ['#4C72B0' if i == 0 else '#aec7e8' for i in range(len(vals))]
    ax.bar(vals.index, vals.values, color=bar_colors, edgecolor='white')
    ax.set_title(metric, fontsize=12)
    ax.set_ylim(0, 1.05)
    ax.tick_params(axis='x', rotation=30)
    for bar, val in zip(ax.patches, vals.values):
        ax.text(bar.get_x() + bar.get_width()/2,
                bar.get_height() + 0.005,
                f'{val:.3f}', ha='center', va='bottom', fontsize=8)

plt.suptitle('Model Comparison — Accuracy / F1 / ROC-AUC', fontsize=14)
plt.tight_layout()
plt.show()

## Step 9.8 — Decision Tree Visualization

A decision tree with limited depth (max_depth=4) is displayed for readability.

In [ ]:
dt_vis = DecisionTreeClassifier(max_depth=4, random_state=RANDOM_SEED)
dt_vis.fit(X_train, y_train)

fig, ax = plt.subplots(figsize=(22, 9))
plot_tree(
    dt_vis,
    feature_names=X_train.columns.tolist(),
    class_names=['Low Intent', 'High Intent'],
    filled=True, rounded=True, fontsize=7, ax=ax,
)
plt.title('Decision Tree (max_depth=4)', fontsize=13)
plt.tight_layout()
plt.show()

print(f'Train accuracy (depth=4): {dt_vis.score(X_train, y_train):.3f}')
print(f'Val   accuracy (depth=4): {dt_vis.score(X_val, y_val):.3f}')

## Step 9.9 — Feature Importance (Random Forest)

Random Forest assigns an importance score to each feature — this reveals which columns have the greatest influence on predictions.

In [ ]:
rf_model = models['Random Forest']

feat_imp = (
    pd.Series(rf_model.feature_importances_, index=X_train.columns)
    .sort_values(ascending=False)
)

fig, ax = plt.subplots(figsize=(13, 5))
feat_imp.head(15).plot(kind='bar', ax=ax, color='steelblue', edgecolor='white')
ax.set_title('Top 15 Feature Importances — Random Forest', fontsize=13)
ax.set_ylabel('Importance')
ax.tick_params(axis='x', rotation=45)
plt.tight_layout()
plt.show()

print('Top 10 features:')
print(feat_imp.head(10).round(4).to_string())

## Step 9.10 — Hyperparameter Tuning (GridSearchCV)

The best model by F1 is tuned. `GridSearchCV` searches over parameter combinations and selects the best.

In [ ]:
best_model_name = eval_df['F1'].idxmax()
print(f'Best model by F1 on validation set: {best_model_name}\n')

# Pipeline-wrapped models (LR, NB, SVM) expose their hyperparameters under
# the 'clf__' prefix so GridSearchCV can reach through the pipeline.
# Tree-based models (DT, RF) are not wrapped, so their params use no prefix.
param_grids = {
    'Logistic Regression': {
        'clf__C'       : [0.01, 0.1, 1, 10],
        'clf__penalty' : ['l2'],
        'clf__solver'  : ['lbfgs'],
        'clf__max_iter': [1000],
    },
    'Decision Tree': {
        'max_depth'        : [5, 10, 20, None],
        'min_samples_split': [2, 5, 10],
        'criterion'        : ['gini', 'entropy'],
    },
    'Random Forest': {
        'n_estimators'    : [100, 300],
        'max_depth'       : [10, 20, None],
        'min_samples_leaf': [1, 5],
    },
    'Naive Bayes': {},
    'SVM': {
        'clf__C'     : [0.1, 1, 10],
        'clf__kernel': ['rbf', 'linear'],
        'clf__gamma' : ['scale', 'auto'],
    },
}

cv_tune = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_SEED)

if param_grids.get(best_model_name):
    base_clf = models[best_model_name]
    grid_search = GridSearchCV(
        base_clf, param_grids[best_model_name],
        cv=cv_tune, scoring='f1', n_jobs=-1, refit=True, verbose=0,
    )
    grid_search.fit(X_train_full, y_train_full)
    best_model_tuned = grid_search.best_estimator_

    print(f'Best hyperparameters for {best_model_name}:')
    for k, v in grid_search.best_params_.items():
        print(f'  {k}: {v}')
    print(f'\nCV F1 before tuning : {eval_df.loc[best_model_name, "F1"]:.4f}')
    print(f'CV F1 after tuning  : {grid_search.best_score_:.4f}')
else:
    best_model_tuned = models[best_model_name]
    best_model_tuned.fit(X_train_full, y_train_full)
    print(f'No grid defined for {best_model_name} — using default parameters.')

## Step 9.11 — Imbalanced Data Handling

Three resampling strategies are compared:

| Method | Description |
|---|---|
| **SMOTE** | Creates synthetic samples for the minority class |
| **ADASYN** | Creates more samples in hard-to-classify regions |
| **RandomUnderSampler** | Reduces the majority class |

Comparison is based on Random Forest.

In [ ]:
if not IMBLEARN_AVAILABLE:
    print('Install imbalanced-learn:  pip install imbalanced-learn')
else:
    smote = SMOTE(random_state=RANDOM_SEED)
    X_smote, y_smote = smote.fit_resample(X_train, y_train)

    print(f'Before SMOTE: {dict(pd.Series(y_train).value_counts().sort_index())}')
    print(f'After  SMOTE: {dict(pd.Series(y_smote).value_counts().sort_index())}')

    rf_before = RandomForestClassifier(n_estimators=100, random_state=RANDOM_SEED, n_jobs=-1)
    rf_before.fit(X_train, y_train)
    y_pred_before = rf_before.predict(X_val)

    rf_after = RandomForestClassifier(n_estimators=100, random_state=RANDOM_SEED, n_jobs=-1)
    rf_after.fit(X_smote, y_smote)
    y_pred_after = rf_after.predict(X_val)

    smote_comparison = pd.DataFrame({
        'Before SMOTE': {
            'Accuracy' : accuracy_score(y_val, y_pred_before),
            'Precision': precision_score(y_val, y_pred_before, zero_division=0),
            'Recall'   : recall_score(y_val, y_pred_before, zero_division=0),
            'F1'       : f1_score(y_val, y_pred_before, zero_division=0),
        },
        'After SMOTE': {
            'Accuracy' : accuracy_score(y_val, y_pred_after),
            'Precision': precision_score(y_val, y_pred_after, zero_division=0),
            'Recall'   : recall_score(y_val, y_pred_after, zero_division=0),
            'F1'       : f1_score(y_val, y_pred_after, zero_division=0),
        },
    }).round(4)

    print('\nRandom Forest — Before vs. After SMOTE:')
    display(smote_comparison)

    fig, ax = plt.subplots(figsize=(8, 5))
    smote_comparison.T.plot(kind='bar', ax=ax,
                             color=['#4C72B0', '#DD8452'], edgecolor='white', width=0.7)
    ax.set_title('Random Forest: Before vs. After SMOTE', fontsize=12)
    ax.set_ylabel('Score')
    ax.set_ylim(0, 1.05)
    ax.tick_params(axis='x', rotation=0)
    ax.legend(loc='lower right')
    plt.tight_layout()
    plt.show()

In [ ]:
if not IMBLEARN_AVAILABLE:
    print('Install imbalanced-learn:  pip install imbalanced-learn')
else:
    sampling_methods = {
        'ADASYN'             : ADASYN(random_state=RANDOM_SEED),
        'RandomUnderSampler' : RandomUnderSampler(random_state=RANDOM_SEED),
    }

    sampling_results = {}
    for method_name, sampler in sampling_methods.items():
        X_res, y_res = sampler.fit_resample(X_train, y_train)
        rf = RandomForestClassifier(n_estimators=100, random_state=RANDOM_SEED, n_jobs=-1)
        rf.fit(X_res, y_res)
        y_pred = rf.predict(X_val)
        sampling_results[method_name] = {
            'Accuracy' : accuracy_score(y_val, y_pred),
            'Precision': precision_score(y_val, y_pred, zero_division=0),
            'Recall'   : recall_score(y_val, y_pred, zero_division=0),
            'F1'       : f1_score(y_val, y_pred, zero_division=0),
        }
        print(f'{method_name}: F1={sampling_results[method_name]["F1"]:.4f}  '
              f'Recall={sampling_results[method_name]["Recall"]:.4f}  '
              f'Samples after resampling: {len(y_res)}')

    sampling_df = pd.DataFrame(sampling_results).T.round(4)
    print()
    display(sampling_df)

## Step 9.12 — Cost-Sensitive Learning

> **Note:** The dataset does not include explicit misclassification costs (i.e., the financial or business cost of a False Negative vs. a False Positive is not quantified). This section was included deliberately to demonstrate that we studied and understand cost-sensitive learning, and that if such cost information were available, it could be incorporated into the model in a straightforward way — either through `class_weight` or by adjusting the decision threshold.

Two approaches:
1. `class_weight='balanced'` — the model assigns higher weight to the minority class
2. **Threshold adjustment** — instead of the default 0.5 threshold, lower values increase Recall

Useful when FN (missing a High Intent user) is more costly than FP.

In [ ]:
rf_balanced = RandomForestClassifier(
    n_estimators=100, class_weight='balanced',
    random_state=RANDOM_SEED, n_jobs=-1,
)
rf_balanced.fit(X_train, y_train)
y_proba_bal = rf_balanced.predict_proba(X_val)[:, 1]

thresholds = [0.3, 0.4, 0.5, 0.6]
threshold_results = {}

for t in thresholds:
    y_pred_t = (y_proba_bal >= t).astype(int)
    threshold_results[f'Threshold={t}'] = {
        'Precision': precision_score(y_val, y_pred_t, zero_division=0),
        'Recall'   : recall_score(y_val, y_pred_t, zero_division=0),
        'F1'       : f1_score(y_val, y_pred_t, zero_division=0),
        'Accuracy' : accuracy_score(y_val, y_pred_t),
    }

threshold_df = pd.DataFrame(threshold_results).T.round(4)
print('Cost-Sensitive Learning — Threshold Analysis (class_weight=balanced):')
display(threshold_df)

print('\nLowering the threshold increases Recall (fewer false negatives) but reduces Precision.')
print('The choice depends on the business cost of FN vs. FP.')

## Step 9.13 — Precision-Recall Curve

With imbalanced data, the PR curve is often more informative than ROC. The red line represents the no-skill baseline.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 6))

for name, model in models.items():
    if hasattr(model, 'predict_proba'):
        y_proba = model.predict_proba(X_val)[:, 1]
        precision_vals, recall_vals, _ = precision_recall_curve(y_val, y_proba)
        ap = average_precision_score(y_val, y_proba)
        ax.plot(recall_vals, precision_vals, lw=2, label=f'{name}  (AP = {ap:.3f})')

no_skill = y_val.mean()
ax.axhline(no_skill, color='red', linestyle='--', lw=1,
           label=f'No-skill  (P = {no_skill:.2f})')
ax.set_xlabel('Recall', fontsize=11)
ax.set_ylabel('Precision', fontsize=11)
ax.set_title('Precision-Recall Curves — All Models', fontsize=13)
ax.legend(loc='upper right', fontsize=9)
plt.tight_layout()
plt.show()

## Step 9.14 — Final Model Evaluation and Test Predictions

The tuned model is retrained on the full training data and predictions are generated for the test set.

In [ ]:
print(f'Selected model: {best_model_name} (tuned)\n')

best_model_tuned.fit(X_train, y_train)
y_pred_final  = best_model_tuned.predict(X_val)
y_proba_final = (
    best_model_tuned.predict_proba(X_val)[:, 1]
    if hasattr(best_model_tuned, 'predict_proba') else None
)

print('=== Classification Report ===')
print(classification_report(y_val, y_pred_final,
                             target_names=['Low Intent (0)', 'High Intent (1)']))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ConfusionMatrixDisplay.from_predictions(
    y_val, y_pred_final,
    display_labels=['Low Intent', 'High Intent'],
    cmap='Blues', ax=axes[0],
)
axes[0].set_title(f'Confusion Matrix — {best_model_name}', fontsize=12)

if y_proba_final is not None:
    fpr, tpr, _ = roc_curve(y_val, y_proba_final)
    auc_val = roc_auc_score(y_val, y_proba_final)
    axes[1].plot(fpr, tpr, color='steelblue', lw=2, label=f'AUC = {auc_val:.3f}')
    axes[1].plot([0, 1], [0, 1], 'k--', lw=1, label='Random')
    axes[1].set_xlabel('False Positive Rate')
    axes[1].set_ylabel('True Positive Rate')
    axes[1].set_title('ROC Curve — Final Model', fontsize=12)
    axes[1].legend()

plt.tight_layout()
plt.show()

In [ ]:
best_model_tuned.fit(X_train_full, y_train_full)

y_pred_test  = best_model_tuned.predict(X_test)
y_proba_test = (
    best_model_tuned.predict_proba(X_test)[:, 1]
    if hasattr(best_model_tuned, 'predict_proba')
    else np.full(len(X_test), float('nan'))
)

predictions_df = pd.DataFrame({
    'predicted_high_intent': y_pred_test,
    'proba_high_intent'    : y_proba_test.round(4),
})
predictions_df.to_csv('PreProcessedData/test_predictions.csv', index=False)

print(f'Saved: PreProcessedData/test_predictions.csv  ({len(predictions_df)} rows)')
print(f'\nPredicted class distribution:')
print(predictions_df['predicted_high_intent'].value_counts().to_string())
print(f'\nMean predicted probability: {y_proba_test.mean():.3f}')
display(predictions_df.head(10))